# 03 - Attention And Transformers Visual

## Learning objectives

- Explain the purpose of query, key, and value vectors.
- Compute a small attention example by hand-sized code.
- Visualize attention weights.
- Connect attention to contextual embeddings inside transformer layers.

## Concept primer

Static embeddings give each token a vector. Transformers make those vectors contextual. The word `bank` should mean something different in `river bank` and `bank loan`. Self-attention is one mechanism that lets each token update its representation using nearby and distant tokens.


## Step 1 - Setup


In [ ]:
# This setup cell keeps imports working whether Jupyter starts in the repo root
# or inside the nlp_transformers_embeddings folder.
from pathlib import Path
import os
import sys

CURRENT = Path.cwd()
COURSE_ROOT = CURRENT.parent if CURRENT.name == "nlp_transformers_embeddings" else CURRENT
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dotenv import load_dotenv
load_dotenv(COURSE_ROOT / ".env", override=False)
load_dotenv(COURSE_ROOT / "nlp_transformers_embeddings" / ".env", override=False)

# Security practice: print whether keys are present, never the key values.
print("LIVE_API enabled:", os.getenv("LIVE_API", "false").lower() == "true")
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("ANTHROPIC_API_KEY loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))


## Step 2 - A tiny attention calculation

In a transformer, every token produces a query, key, and value. The query asks, `what am I looking for?` Keys advertise, `what information do I contain?` Values carry the information to mix into the output.


In [ ]:
import numpy as np
from nlp_transformers_embeddings.utils.viz import scaled_dot_product_attention, plot_attention

# These tiny vectors are hand-made for teaching. Real transformer weights are learned.
tokens = ["RAG", "retrieves", "evidence"]
query_for_retrieves = np.array([0.2, 0.9, 0.1])
keys = np.array([
    [0.1, 0.2, 0.8],  # RAG
    [0.2, 0.8, 0.1],  # retrieves
    [0.1, 0.7, 0.6],  # evidence
])
values = np.array([
    [1.0, 0.0, 0.2],
    [0.2, 1.0, 0.1],
    [0.3, 0.4, 1.0],
])

# Scores show raw compatibility; weights show normalized attention after softmax.
scores, weights, output = scaled_dot_product_attention(query_for_retrieves, keys, values)

print("Raw attention scores:", scores.round(3))
print("Attention weights:", weights.round(3))
print("Contextual output vector:", output.round(3))
plot_attention(tokens, weights, title="How 'retrieves' attends to the sentence");


## What just happened?

The token `retrieves` mixed information from all tokens, weighted by compatibility. This is the heart of self-attention: token representations become context-aware rather than fixed dictionary entries.


## Step 3 - Positional signal intuition

Attention alone sees a set of tokens. Position information tells the model that `dog bites man` and `man bites dog` are different.


In [ ]:
import math

def sinusoidal_position(position, dimension=6):
    # This small version mirrors the classic transformer positional encoding idea.
    values = []
    for i in range(dimension):
        angle = position / (10000 ** ((2 * (i // 2)) / dimension))
        values.append(math.sin(angle) if i % 2 == 0 else math.cos(angle))
    return values

for pos in range(4):
    print(f"position {pos}:", [round(v, 3) for v in sinusoidal_position(pos)])


## Step 4 - Where OpenAI and Anthropic fit

OpenAI and Anthropic both serve transformer-family language models. You generally do not manipulate Q/K/V directly through their APIs. Instead, you control inputs, retrieved context, tool calls, schemas, and evaluation. This notebook gives students the internal mental model that explains why prompt order, context selection, and chunk boundaries matter.


## Student exercise

Change the query vector and observe how the attention weights shift. Which token becomes most influential? Why?

## Debugging checklist

- If the plot does not render, run the cell again after `%matplotlib inline`.
- If vector shapes fail, check that query, keys, and values have compatible dimensions.

## Production best practices

- Do not overinterpret attention weights from production models as complete explanations.
- Use this mental model to design better context, not to claim perfect interpretability.
- Put the most relevant evidence into the context window; attention cannot use evidence that was never supplied.

## Reflection questions

1. Why do transformer embeddings become contextual?
2. Why does position matter in language?
3. How does this lesson explain the importance of RAG chunk design?
